In [1]:
import pandas as pd

In [2]:
mortalitas_df = pd.read_csv("./data/mortalitas.csv", parse_dates=["Year"], date_format="%Y")
populasi_df = pd.read_csv("./data/populasi.csv")
bi_rate_df = pd.read_csv("./data/bi_rate.csv", parse_dates=["Date"], date_format="%d-%m-%y")

In [3]:
mortalitas_df.head()

,Year,Sex,Age,Value
0,1950-01-01,Male,0,0.228515
1,1950-01-01,Female,0,0.185158
2,1950-01-01,Male,1,0.069349
3,1950-01-01,Female,1,0.063058
4,1950-01-01,Male,2,0.044724


In [4]:
mortalitas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15150 entries, 0 to 15149
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Year    15150 non-null  datetime64[us]
 1   Sex     15150 non-null  str           
 2   Age     15150 non-null  int64         
 3   Value   15150 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 473.6 KB


In [5]:
populasi_df.head()

,Sex,Age,Value
0,Male,45,2013106
1,Male,46,1965078
2,Male,47,1918556
3,Male,48,1880902
4,Male,49,1836713


In [6]:
populasi_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Sex     32 non-null     str  
 1   Age     32 non-null     int64
 2   Value   32 non-null     int64
dtypes: int64(2), str(1)
memory usage: 900.0 bytes


In [7]:
bi_rate_df.head()

,Date,Value
0,2024-12-18,0.0600
1,2024-11-20,0.0600
2,2024-10-16,0.0600
3,2024-09-18,0.0600
4,2024-08-21,0.0625


In [8]:
bi_rate_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    60 non-null     datetime64[us]
 1   Value   60 non-null     float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 1.1 KB


In [9]:
AGE_MIN = min(mortalitas_df["Age"])
AGE_MAX = max(mortalitas_df["Age"])

YEAR_MIN = min(mortalitas_df["Year"])
YEAR_MAX = max(mortalitas_df["Year"])

# Analisis karakteristik data

In [10]:
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid")

## Plot mortalitas satu usia untuk semua tahun

In [11]:
def plot_usia_vs_tahun(
        age_start=AGE_MIN,
        age_end=AGE_MAX
):
    plotname = f"usia vs tahun ({age_start}-{age_end})"
    filepath = Path.cwd() / "plot" / f"{plotname}.png"
    if not filepath.exists():
        mask = (mortalitas_df["Age"] >= age_start) & (mortalitas_df["Age"] <= age_end)
        g = sns.FacetGrid(
            mortalitas_df[mask],
            col="Age",
            hue="Sex",
            height=5,
            col_wrap=5,
            sharex=False,

            sharey=False
        )

        g.map_dataframe(sns.lineplot, x="Year", y="Value")

        g.set_titles("Age {col_name}")
        g.set_axis_labels("Year", "Mortality Rate")
        g.add_legend()

        g.tight_layout()
        g.savefig(filepath)
        print(f"Plot {plotname} saved to {filepath}")

In [12]:
plot_usia_vs_tahun(AGE_MIN, 20)
plot_usia_vs_tahun(21, 40)
plot_usia_vs_tahun(41, 60)
plot_usia_vs_tahun(61, 80)
plot_usia_vs_tahun(81, AGE_MAX)

## Plot mortalitas semua usia untuk satu tahun

In [13]:
def plot_tahun_vs_usia(
        year_start=YEAR_MIN,
        year_end=YEAR_MAX
):
    plotname = f"tahun vs usia ({year_start}-{year_end})"
    filepath = Path.cwd() / "plot" / f"{plotname}.png"
    if not filepath.exists():
        mask = (mortalitas_df["Year"] >= year_start) & (mortalitas_df["Year"] <= year_end)
        df = mortalitas_df.copy()
        df["Year Only"] = df["Year"].dt.year

        g = sns.FacetGrid(
            df[mask],
            col="Year Only",
            hue="Sex",
            height=5,
            col_wrap=5,
            sharex=False,
            sharey=False
        )

        g.map_dataframe(sns.lineplot, x="Age", y="Value")

        g.set_titles("Year {col_name}")
        g.set_axis_labels("Age", "Mortality Rate")
        g.add_legend()

        g.tight_layout()
        g.savefig(filepath)
        print(f"Plot {plotname} saved to {filepath}")

In [14]:
plot_tahun_vs_usia("1950", "1964")
plot_tahun_vs_usia("1965", "1979")
plot_tahun_vs_usia("1980", "1994")
plot_tahun_vs_usia("1995", "2009")
plot_tahun_vs_usia("2010", "2024")

# Pemodelan LocalGLMnet

In [15]:
import torch
import copy
import torch.nn as nn
import numpy as np

from typing import Callable, Iterator, Collection, Any
from torch import Tensor
from torch.nn import Parameter
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Transform
from torch.distributions.constraints import real

from pandas import DataFrame
from pandas.api.types import is_datetime64_any_dtype
from math import floor, ceil

## Dependencies untuk pemodelan

In [16]:
class MortalityDataset(Dataset):

    def __init__(
            self,
            lookback: int,
            horizon: int,
            df: DataFrame,
            mortality_col: str,
            age_col: str,
            year_col: str
    ):
        if lookback <= 0:
            raise ValueError("lookback harus > 0")
        if horizon <= 0:
            raise ValueError("horizon harus > 0")
        if df.isna().any(axis=None):
            raise ValueError("Terdapat missing value pada df")
        if not is_datetime64_any_dtype(df[year_col]):
            raise TypeError("df[year_col] harus tipe datetime-like")

        self.df_pivoted = df.pivot(
            columns=age_col,
            index=year_col,
            values=mortality_col
        )

        self.lookback = lookback
        self.horizon = horizon

    def __len__(self):
        return len(self.df_pivoted) - self.lookback - self.horizon + 1

    def __getitem__(self, idx: int):
        n = self.__len__()
        l = self.lookback
        h = self.horizon

        if idx > 0 and idx >= n:
            raise ValueError(f"idx positif hanya valid di [{0}, {n})")

        if idx < 0 and -idx > n:
            raise ValueError(f"idx negatif hanya valid di [{-n}, 0)")

        x_ind = np.arange(idx, idx + l) if idx >= 0 \
            else np.arange(n + idx * l - 1, n + idx * l + h - 1)

        y_ind = np.arange(idx + l, idx + l + h) if idx >= 0 \
            else np.arange(n + (idx + 1) * l - 1, n + (idx + 1) * l + h - 1)

        x = self.df_pivoted.iloc[x_ind, :].to_numpy(copy=True, dtype=np.float32)
        y = self.df_pivoted.iloc[y_ind, :].to_numpy(copy=True, dtype=np.float32)

        return torch.from_numpy(x), torch.from_numpy(y)

In [17]:
def ts_collate(batch: list[Any]) -> tuple[Tensor, Tensor]:
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ys)

In [18]:
class LocallyConnected2D(nn.Module):
    def __init__(
            self,
            input_size: int | tuple[int, int],
            kernel_size: int,
            activation_fn: nn.Module,
            stride: int = 1,
            dilation: int = 1,
            zero_padding: bool = False,
            bias: bool = True
    ):
        super().__init__()

        self.stride = stride
        self.dilation = dilation
        self.kernel_size = kernel_size
        self.zero_padding = zero_padding
        self.activation_fn = activation_fn

        if isinstance(input_size, int):
            self.H_in, self.W_in = (input_size, input_size)
        else:
            self.H_in, self.W_in = input_size

        self.pad = (kernel_size - 1) / 2 if zero_padding else 0

        self.H_out: int = floor((self.H_in + 2 * self.pad - dilation * (kernel_size - 1) - 1) / stride + 1)
        self.W_out: int = floor((self.W_in + 2 * self.pad - dilation * (kernel_size - 1) - 1) / stride + 1)

        self.weight = nn.Parameter(
            data=torch.rand(
                self.H_out, self.W_out, kernel_size, kernel_size,
                dtype=torch.float32
            ))

        if bias:
            self.bias = nn.Parameter(
                data=torch.rand(
                    self.H_out, self.W_out,
                    dtype=torch.float32
                ))
        else:
            self.register_parameter('bias', None)

    def forward(self, x: Tensor[any, any]):
        unbatches = x.dim() == 2
        if unbatches:
            x = x.unsqueeze(0)

        N = x.size(0)
        k = self.kernel_size

        # Ubah dimensi x menjadi (N, 1, x_dim0, x_dim1): menambahkan dimensi untuk channel (C)
        x = x.unsqueeze(1)

        # Tambahkan padding 0 di sisi luar x
        if self.pad > 0:
            x = nn.functional.pad(
                x,
                [ceil(self.pad), floor(self.pad), ceil(self.pad), floor(self.pad)]
            )

        # Membuat tensor untuk diproses oleh kernel
        # Dimensi tensor (N, k*k, H_out*W_out)
        patches = nn.functional.unfold(x, kernel_size=k)

        # Ubah dimensi patches menjadi (N, k, k, H_out, W_out)
        patches = patches.view(N, k, k, self.H_out, self.W_out)

        # Dimensi patches (N, k, k, H_out, W_out): indeks (n, k, l, h, w)
        # Dimensi weight (H_out, W_out, k, k): indeks (h, w, k, l)
        # Sum pada indeks k dan l
        # Output tensor punya dimensi (N, H_out, W_out)
        y = torch.einsum("nklhw, hwkl -> nhw", patches, self.weight)

        # Tambahkan bias
        if self.bias is not None:
            # Ubah dimensi bias menjadi (N, H_out, W_out) sebelum dijumlahkan
            y = y + self.bias.unsqueeze(0).expand(N, -1, -1)

        if unbatches:
            y = y.squeeze(0)

        return self.activation_fn(y)

    @classmethod
    def factory(
            cls,
            input_size: int | tuple[int, int],
            kernel_size: int,
            activation_fn: nn.Module,
            stride: int = 1,
            dilation: int = 1,
            zero_padding: bool = False,
            bias: bool = True
    ):
        def create():
            return cls(
                input_size,
                kernel_size,
                activation_fn,
                stride,
                dilation,
                zero_padding,
                bias
            )

        return create

    def extra_repr(self):
        name = "LocallyConnected2D"
        args = "\n".join([
            "================================================================================",
            "Arguments:",
            f"input_size        = ({self.H_in}, {self.W_in})",
            f"kernel_size       = ({self.kernel_size}, {self.kernel_size})",
            f"stride            = {self.stride}",
            f"dilation          = {self.dilation}",
            f"zero_padding      = {self.zero_padding}",
            f"activation_fn     = {self.activation_fn}",
        ])
        params = "\n".join([
            "================================================================================",
            "Parameters:",
            f"weight = {self.weight.numel()} trainable parameters",
            f"bias   = {self.bias.numel()} trainable parameters",
        ])

        return (
            f"{name}\n"
            f"{args}\n"
            f"{params}\n"
            "================================================================================\n"
            f"Output size = ({self.H_out}, {self.W_out})\n"
        )

In [19]:
class IdentityTransform(Transform):
    domain = real
    codomain = real
    bijective = True

    def _call(self, x):
        return x

    def _inverse(self, y):
        return y

In [20]:
class LocalGLMnet(nn.Module):
    def __init__(self,
                 input_size: tuple[int, int],
                 link_fn: Transform,
                 regression_attention_model: nn.Module,
                 bias: bool = True):
        super().__init__()
        self.input_size = input_size
        self.output_size = input_size[1]

        self.link_fn = link_fn
        self.regression_attention_model = regression_attention_model

        if bias:
            self.bias = nn.Parameter(data=torch.rand(self.output_size))
        else:
            self.register_parameter('bias', None)

    def forward(self, x: Tensor):
        if x.size()[1:] != self.input_size:
            raise ValueError(
                f"Size x={x.size()[1:]} tidak sama dengan input_size={self.input_size}: Ganti nilai x yang punya size={self.input_size}")

        regression_attention: Tensor = self.regression_attention_model(x)
        if regression_attention.size()[1:] != self.input_size:
            raise AttributeError(
                f"Size dari regression_atention_model(x) {self.regression_attention.size()[1:]} tidak sama dengan input_size={self.input_size}: Definisikan ulang LocalGLMnet dengan regression_attention_model yang menghasilkan output_size sama dengan input_size-nya")
        unbatches = x.dim() == 2
        if unbatches:
            x = x.unsqueeze(0).expand(regression_attention.size()[0], -1, -1)

        # regression_attention punya dimensi (N, H, W)
        # x punya dimensi (H, W)
        # Hadamard product untuk regression_attention dan x, punya dimensi (N, H, W)
        # Dimensi x akan dibroadcast menjadi (N, H, W)
        w_hadamard_x = regression_attention * x

        # self.bias punya dimensi (W)
        # self.bias akan dibroadcast menjadi (N, W)
        # y punya dimensi (N, W)
        y = self.link_fn.inv(
            w_hadamard_x.sum(dim=1) + self.bias
        )

        # Ubah dimensi y menjadi (N, 1, W) agar konsisten dengan MortalityDataset
        y = y.unsqueeze(1)

        return y

    @classmethod
    def factory(
            cls,
            input_size: tuple[int, int],
            link_fn: Transform,
            regression_attention_model: nn.Module,
            bias: bool = True
    ):
        def create():
            return cls(input_size, link_fn, regression_attention_model, bias)

        return create

        def extra_repr(self) -> str:
            name = "LocalGLMnet"
            args = "\n".join([
                "================================================================================",
                "Arguments:",
                f"input_size    = ({self.input_size[0]}, {self.input_size[1]})",
                f"link_fn       = {self.link_fn}",
                f"bias          = {self.bias != None}"
            ])
            params = "\n".join([
                "================================================================================",
                "Parameters:",
                f"bias      = {self.bias.numel()} trainable parameters",
            ])

            return (
                f"{name}\n"
                f"{args}\n"
                f"{params}\n"
                "================================================================================\n"
                f"Output size = ({self.output_size})\n"
            )

In [21]:
class LASSOLoss(nn.Module):
    def __init__(self,
                 base_loss_fn: Callable[[Tensor, Tensor], Tensor],
                 regularization_parameter: float,
                 model_weights: Iterator[Parameter]):
        super().__init__()
        self.base_loss_fn = base_loss_fn
        self.regularization_parameter = regularization_parameter
        self.model_weights = list(model_weights)

    def forward(self, pred: Tensor, target: Tensor) -> Tensor:
        base_loss: Tensor = self.base_loss_fn(pred, target)
        l1_term: Tensor = sum(w.abs().sum() for w in self.model_weights) / len(target)

        return base_loss + (self.regularization_parameter * l1_term)

    @classmethod
    def factory(
            cls,
            base_loss_fn: Callable[[Tensor, Tensor], Tensor],
            regularization_parameter: float,
            model_weights: Iterator[Parameter]
    ):
        def create():
            return cls(base_loss_fn, regularization_parameter, model_weights)

        return create

In [23]:
def train_ensembles(
        n_epochs: int,
        models: nn.ModuleList | Collection[nn.Module],
        train_dataloaders: Collection[DataLoader],
        optimizers: torch.optim.Optimizer | Collection[torch.optim.Optimizer],
        loss_fns: nn.Module | Collection[nn.Module],
        device: str = "cpu"
):
    n_models = len(models)

    # Normalize optimizers
    if isinstance(optimizers, torch.optim.Optimizer):
        # Satu optimizer untuk semua model — wrap jadi list
        optimizers = [optimizers]
    if len(optimizers) != n_models:
        raise ValueError(
            f"Jumlah optimizers ({len(optimizers)}) harus sama dengan jumlah models ({n_models})"
        )

    # Normalize loss_fns
    if isinstance(loss_fns, nn.Module):
        # Satu loss fn untuk semua model — deepcopy per model
        loss_fns = [copy.deepcopy(loss_fns) for _ in range(n_models)]
    if len(loss_fns) != n_models:
        raise ValueError(
            f"Jumlah loss_fns ({len(loss_fns)}) harus sama dengan jumlah models ({n_models})"
        )

    for m in models:
        m.to(device)
        m.train()

    for epoch in range(1, n_epochs + 1):
        print(f"{'=' * 64}")
        print(f"Epoch {epoch}/{n_epochs}")

        for i, dl in enumerate(train_dataloaders):
            print(f"\n  Dataloader {i + 1}/{len(train_dataloaders)}:")
            epoch_loss = 0.0

            for batch_idx, (x, y) in enumerate(dl):
                x = x.to(device)
                y = y.to(device)

                # Zero grad di awal tiap batch
                for opt in optimizers:
                    opt.zero_grad()

                # Backward per model — gradient independen
                losses = []
                for m, loss_fn in zip(models, loss_fns):
                    pred = m(x)
                    loss = loss_fn(pred, y)
                    losses.append(loss)

                for loss in losses:
                    loss.backward()

                # Step setelah semua backward
                for opt in optimizers:
                    opt.step()

                avg_loss = sum(l.item() for l in losses) / n_models
                epoch_loss += avg_loss
                print(f"    Batch {batch_idx + 1}: avg loss = {avg_loss:.4f}")

            print(f"  Epoch loss = {epoch_loss / len(dl):.4f}")

In [24]:
def evaluate_ensembles(
        models: nn.ModuleList | Collection[nn.Module],
        test_dataloaders: Collection[DataLoader],
        loss_fns: nn.Module | Collection[nn.Module],
        device: str = "cpu"
) -> dict:
    n_models = len(models)

    # Normalize loss_fns
    if isinstance(loss_fns, nn.Module):
        loss_fns = [copy.deepcopy(loss_fns) for _ in range(n_models)]
    if len(loss_fns) != n_models:
        raise ValueError(
            f"Jumlah loss_fns ({len(loss_fns)}) harus sama dengan jumlah models ({n_models})"
        )

    for m in models:
        m.to(device)
        m.eval()

    results = {
        "per_model": [{} for _ in range(n_models)],
        "ensemble": {}
    }

    with torch.no_grad():
        for i, dl in enumerate(test_dataloaders):
            print(f"{'=' * 64}")
            print(f"Dataloader {i + 1}/{len(test_dataloaders)}:")

            # Kumpulkan semua prediksi dan target
            all_preds = [[] for _ in range(n_models)]
            all_targets = []

            for x, y in dl:
                x = x.to(device)
                y = y.to(device)
                all_targets.append(y)

                for j, m in enumerate(models):
                    pred = m(x)
                    all_preds[j].append(pred)

            # Concat semua batch
            all_targets = torch.cat(all_targets, dim=0)
            all_preds = [torch.cat(preds, dim=0) for preds in all_preds]

            # Loss per model
            dl_key = f"dataloader_{i + 1}"
            for j, (preds, loss_fn) in enumerate(zip(all_preds, loss_fns)):
                loss = loss_fn(preds, all_targets).item()
                results["per_model"][j][dl_key] = loss
                print(f"  Model {j + 1}: loss = {loss:.4f}")

            # Loss ensemble (rata-rata prediksi)
            ensemble_preds = torch.stack(all_preds).mean(dim=0)
            ensemble_loss = loss_fns[0](ensemble_preds, all_targets).item()
            results["ensemble"][dl_key] = ensemble_loss
            print(f"  Ensemble  : loss = {ensemble_loss:.4f}")

    return results

## Pemodelan

In [25]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [26]:
lookback = 10
horizon = 1

mortalitas_dataset_male = MortalityDataset(
    df=mortalitas_df[mortalitas_df["Sex"] == "Male"],
    lookback=lookback,
    horizon=horizon,
    age_col="Age",
    mortality_col="Value",
    year_col="Year"
)

mortalitas_dataset_female = MortalityDataset(
    df=mortalitas_df[mortalitas_df["Sex"] == "Female"],
    lookback=lookback,
    horizon=horizon,
    age_col="Age",
    mortality_col="Value",
    year_col="Year"
)

mortalitas_dataloader_male = DataLoader(
    dataset=mortalitas_dataset_male,
    batch_size=len(mortalitas_dataset_male),
    collate_fn=ts_collate,
    shuffle=True
)

mortalitas_dataloader_female = DataLoader(
    dataset=mortalitas_dataset_female,
    batch_size=len(mortalitas_dataset_female),
    collate_fn=ts_collate,
    shuffle=True
)

In [27]:
ukuran_matriks_input = (lookback, AGE_MAX - AGE_MIN + 1)
lcn_activation_function = nn.Sigmoid()
lcn_kernel_size = 5

create_lcn_layer = LocallyConnected2D.factory(
    input_size=ukuran_matriks_input,
    kernel_size=lcn_kernel_size,
    activation_fn=lcn_activation_function,
    zero_padding=True
)

link_function = IdentityTransform()
create_localglmnet_model = LocalGLMnet.factory(
    input_size=ukuran_matriks_input,
    link_fn=link_function,
    regression_attention_model=create_lcn_layer()
)

num_ensembles = 10
localglmnet_models = nn.ModuleList([create_localglmnet_model() for _ in range(num_ensembles)])

lasso_regularization_parameter = 10e-5
base_loss_fn = nn.MSELoss()
loss_fns = [
    LASSOLoss(
        base_loss_fn=base_loss_fn,
        regularization_parameter=lasso_regularization_parameter,
        model_weights=m.regression_attention_model.parameters()
    )
    for m in localglmnet_models
]

optimizers = [
    torch.optim.NAdam(
        params=m.parameters()
    )
    for m in localglmnet_models
]

n_epochs = 500
train_ensembles(
    n_epochs=n_epochs,
    models=localglmnet_models,
    train_dataloaders=[mortalitas_dataloader_male, mortalitas_dataloader_female],
    optimizers=optimizers,
    loss_fns=loss_fns,
    device=device,
)

Epoch 1/500

  Dataloader 1/2:
    Batch 1: avg loss = 3.1948
  Epoch loss = 3.1948

  Dataloader 2/2:
    Batch 1: avg loss = 2.6722
  Epoch loss = 2.6722
Epoch 2/500

  Dataloader 1/2:
    Batch 1: avg loss = 3.1504
  Epoch loss = 3.1504

  Dataloader 2/2:
    Batch 1: avg loss = 2.6330
  Epoch loss = 2.6330
Epoch 3/500

  Dataloader 1/2:
    Batch 1: avg loss = 3.1058
  Epoch loss = 3.1058

  Dataloader 2/2:
    Batch 1: avg loss = 2.5855
  Epoch loss = 2.5855
Epoch 4/500

  Dataloader 1/2:
    Batch 1: avg loss = 3.0472
  Epoch loss = 3.0472

  Dataloader 2/2:
    Batch 1: avg loss = 2.5213
  Epoch loss = 2.5213
Epoch 5/500

  Dataloader 1/2:
    Batch 1: avg loss = 2.9649
  Epoch loss = 2.9649

  Dataloader 2/2:
    Batch 1: avg loss = 2.4299
  Epoch loss = 2.4299
Epoch 6/500

  Dataloader 1/2:
    Batch 1: avg loss = 2.8432
  Epoch loss = 2.8432

  Dataloader 2/2:
    Batch 1: avg loss = 2.2953
  Epoch loss = 2.2953
Epoch 7/500

  Dataloader 1/2:
    Batch 1: avg loss = 2.6573
  

In [28]:
evaluate_results = evaluate_ensembles(
    models=localglmnet_models,
    test_dataloaders=[mortalitas_dataloader_male, mortalitas_dataloader_female],
    loss_fns=nn.MSELoss(),
    device=device,
)

Dataloader 1/2:
  Model 1: loss = 0.0006
  Model 2: loss = 0.0006
  Model 3: loss = 0.0006
  Model 4: loss = 0.0005
  Model 5: loss = 0.0006
  Model 6: loss = 0.0006
  Model 7: loss = 0.0005
  Model 8: loss = 0.0006
  Model 9: loss = 0.0006
  Model 10: loss = 0.0005
  Ensemble  : loss = 0.0004
Dataloader 2/2:
  Model 1: loss = 0.0009
  Model 2: loss = 0.0010
  Model 3: loss = 0.0011
  Model 4: loss = 0.0007
  Model 5: loss = 0.0008
  Model 6: loss = 0.0007
  Model 7: loss = 0.0008
  Model 8: loss = 0.0008
  Model 9: loss = 0.0010
  Model 10: loss = 0.0008
  Ensemble  : loss = 0.0007


In [30]:
evaluate_results["ensemble"]

{'dataloader_1': 0.0003653925086837262, 'dataloader_2': 0.0006837219698354602}

In [35]:
evaluate_results["per_model"]

{'dataloader_1': 0.0005502587300725281, 'dataloader_2': 0.0009404215961694717}